In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
import sys
import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import tqdm
from prophet import Prophet

logger = logging.getLogger('cmdstanpy')
logger.addHandler(logging.NullHandler())
logger.propagate = False
logger.setLevel(logging.CRITICAL)

pd.options.mode.chained_assignment = None

In [ ]:
project_path = "/kaggle/input/store-sales-time-series-forecasting/"

df_stores = pd.read_csv(project_path + "stores.csv")
df_oil = pd.read_csv(project_path + "oil.csv", parse_dates=["date"])
df_holidays = pd.read_csv(project_path + "holidays_events.csv", parse_dates=["date"])
df_transactions = pd.read_csv(project_path + "transactions.csv", parse_dates=["date"])

df = pd.read_csv(project_path + "train.csv", parse_dates=["date"])
df_test = pd.read_csv(project_path + "test.csv", parse_dates=["date"])

In [ ]:
FIRST_DATE = df["date"].min()
LAST_DATE = df_test["date"].max()
VAL_DATE = "2017-08-01"
ALL_DATES = pd.date_range(FIRST_DATE, LAST_DATE)
EARTHQUAKE_DATE = "2016-04-16"
N_STORES = df["store_nbr"].nunique()
N_FAMILIES = df["family"].nunique()

VALIDATION = False
USE_DATA_HOLIDAYS = True
ADD_PROPHET_HOLIDAYS = False
REGRESSORS = ["oil_price", "onpromotion"]

In [ ]:
def get_store_holidays(store_number):

    city_state = df_stores.loc[df_stores["store_nbr"]==store_number, ["city", "state"]].iloc[0]
    df_store_local_holidays = df_holidays[df_holidays["locale_name"].isin(city_state)]
    df_store_national_holidays = df_holidays[df_holidays["locale"]=="National"]
    df_store_holidays = pd.concat([df_store_local_holidays, df_store_national_holidays])
    
    return df_store_holidays[~df_store_holidays["transferred"]].sort_values("date")

In [ ]:
def get_model(holidays=None, add_prophet_holidays=False, regressors=[]):
    
    if holidays is not None:
        model = Prophet(holidays=holidays.rename({"date": "ds", "description": "holiday"}, axis=1))
    else:
        model = Prophet()
    if add_prophet_holidays:
        model.add_country_holidays(country_name="Ecuador")
    if regressors:
        for regressor in regressors:
            model.add_regressor(regressor)
    return model

In [ ]:
def create_and_fit_models(data, use_data_holidays=False, 
                          add_prophet_holidays=False,
                          regressors=[]):

    models = {}
    
    for store_nbr, df_store in data.groupby("store_nbr"):
        df_store_holidays = None
        if use_data_holidays:
            df_store_holidays = get_store_holidays(store_nbr)
        for family, df_store_family in tqdm(df_store.groupby("family"), file=sys.stdout, 
                                            desc=f"Store {store_nbr}/{N_STORES}"):
            model = get_model(df_store_holidays, 
                              add_prophet_holidays=add_prophet_holidays, 
                              regressors=regressors)
            model.fit(df_store_family.rename({"date": "ds", "sales": "y"}, axis=1))
            models[(store_nbr, family)] = model

    return models

In [ ]:
def predict_from_models(data, models):
    
    df_list = []
    
    for store_nbr, df_store in data.groupby("store_nbr"):
        for family, df_store_family in tqdm(df_store.groupby("family"), file=sys.stdout, 
                                            desc=f"Store {store_nbr}/{N_STORES}"):
            model = models[(store_nbr, family)]
            forecast = model.predict(df_store_family.rename(columns={"date": "ds", "sales": "y"}))
            df_store_family["sales_pred"] = forecast["yhat"].clip(lower=0).values
            df_store_family["sales_lower_pred"] = forecast["yhat_lower"].clip(lower=0).values
            df_store_family["sales_upper_pred"] = forecast["yhat_upper"].clip(lower=0).values
            df_list.append(df_store_family)
    
    return pd.concat(df_list).sort_values("id")

In [ ]:
def RMSLE(y_true, y_pred):
    
    if isinstance(y_true, pd.Series):
        y_true = y_true.to_numpy()
    if isinstance(y_pred, pd.Series):
        y_pred = y_pred.to_numpy()
        
    return np.sqrt(np.mean((np.log(1 + y_pred) - np.log(1 + y_true))**2))

In [ ]:
df_stores.head(10)

In [ ]:
df_oil = df_oil.rename({"dcoilwtico": "oil_price"}, axis=1)
df_oil = df_oil.set_index("date").reindex(ALL_DATES).reset_index(names="date")
df_oil["oil_price"] = df_oil["oil_price"].interpolate(limit_direction="both")

df_oil

In [ ]:
df_oil.plot(x="date", y="oil_price", ylabel="oil price",
            title="Oil price over time", figsize=(15, 10));

In [ ]:
df_transactions

In [ ]:
df_holidays.head(10)

In [ ]:
df = pd.merge(df, df_oil, on="date", how="left")

df

In [ ]:
df_test = pd.merge(df_test, df_oil, on="date", how="left")

df_test

In [ ]:
df_sales_per_store = df.groupby(["date", "store_nbr"])["sales"].sum().reset_index()\
.pivot(index="date", columns="store_nbr", values="sales")

df_sales_per_store

In [ ]:
df_sales_per_store[range(1,11)].resample('W').mean()\
.plot(figsize=(15,10), ylabel="sales", title= "Sales per store");

In [ ]:
df_sales_per_family = df.groupby(["date", "family"])["sales"].sum().reset_index()\
.pivot(index="date", columns="family", values="sales")

df_sales_per_family

In [ ]:
df_sales_per_family.resample('W').mean()\
.plot(figsize=(15,10), ylabel="sales", title= "Sales per family")
plt.show()

In [ ]:
store_nbr = 3 #np.random.randint(N_STORES)
family = "BEAUTY" #df["family"].unique()[np.random.randint(N_FAMILIES)]

df_store_family = df[(df["store_nbr"]==store_nbr) & (df["family"]==family)]

df_store_family

In [ ]:
df_store_family.set_index("date")["sales"].resample('W').mean()\
.plot(figsize=(12, 7), ylabel="sales", 
      title=f"Sales for store {store_nbr} and family {family}");

In [ ]:
df_store_holidays = get_store_holidays(store_nbr)

df_store_holidays.head(10)

In [ ]:
df_train = df_store_family[df_store_family["date"] < VAL_DATE]
df_val = df_store_family[df_store_family["date"] >= VAL_DATE]

df_val

In [ ]:
if USE_DATA_HOLIDAYS:
    model = get_model(df_store_holidays, 
                      add_prophet_holidays=ADD_PROPHET_HOLIDAYS, 
                      regressors=REGRESSORS)
else:
    model = get_model(add_prophet_holidays=ADD_PROPHET_HOLIDAYS, 
                      regressors=REGRESSORS)
model.fit(df_train.rename({"date": "ds", "sales": "y"}, axis=1));

In [ ]:
model.train_holiday_names

In [ ]:
forecast = model.predict(df_store_family.rename(columns={"date": "ds", "sales": "y"}))
forecast[["yhat", 'yhat_lower', "yhat_upper"]].clip(lower=0, inplace=True)
forecast_val = forecast[forecast["ds"] >= VAL_DATE]

In [ ]:
model.plot(forecast)
plt.plot(df_val["date"], df_val["sales"], 'r.', label="New data points")
plt.xlim(pd.to_datetime("2017-06-01"), pd.to_datetime("2017-08-15"))
plt.title("Data and Predictions"); plt.xlabel("date"); plt.ylabel("sales")
plt.legend();

In [ ]:
model.plot_components(forecast);

In [ ]:
RMSLE(df_val["sales"], forecast_val["yhat"])

In [ ]:
if VALIDATION:

    df_train = df[df["date"] < VAL_DATE]
    df_val = df[df["date"] >= VAL_DATE]

    display(df_val)

In [ ]:
if VALIDATION:

    models = create_and_fit_models(df_train, 
                                   use_data_holidays=USE_DATA_HOLIDAYS, 
                                   add_prophet_holidays=ADD_PROPHET_HOLIDAYS, 
                                   regressors=REGRESSORS)

In [ ]:
if VALIDATION:

    df_pred = predict_from_models(df_val, models)

In [ ]:
if VALIDATION:

    display(df_pred)

In [ ]:
if VALIDATION:
    
    print(RMSLE(df_pred["sales"], df_pred["sales_pred"]))

In [ ]:
%%time

models = create_and_fit_models(df, use_data_holidays=USE_DATA_HOLIDAYS, 
                               add_prophet_holidays=ADD_PROPHET_HOLIDAYS,
                               regressors=REGRESSORS)

In [ ]:
%%time

df_pred = predict_from_models(df_test, models)

In [ ]:
df_pred

In [ ]:
df_submission = pd.read_csv(project_path + "sample_submission.csv")

df_submission["sales"] = df_pred["sales_pred"]

df_submission

In [ ]:
df_submission.to_csv("submission.csv", index=False)